# Carbon-Aware FL: FedAvg baseline on Colab

Five simulated clients, MNIST, FedAvg, IID or Dirichlet non-IID splits, with round-level and client-level CSV logs.
No scheduler or RL yet; this is the reference run everything later is compared against.

**Runtime:** works on CPU; for speed choose *Runtime → Change runtime type → T4 GPU*.

In [ ]:
# Clone (or update) the repository
REPO_URL = "https://github.com/Harshil089/Carbon-Aware-Federated-Learning.git"
BRANCH = "main"

import os
if not os.path.exists("/content/Carbon-Aware-Federated-Learning"):
    !git clone -b {BRANCH} {REPO_URL} /content/Carbon-Aware-Federated-Learning
else:
    !git -C /content/Carbon-Aware-Federated-Learning pull
%cd /content/Carbon-Aware-Federated-Learning

In [ ]:
!pip install -q -r requirements.txt
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

In [ ]:
# Unit tests for FedAvg and the partitioners (a few seconds)
!python -m pytest -q tests

## Run the baseline

Defaults live in `configs/experiment.yaml` (MNIST, small CNN, 5 clients, all participate, 20 rounds).
Flags override them: `--partition iid|dirichlet`, `--alpha`, `--rounds`, `--clients-per-round`, `--dataset fashion_mnist`, `--model mlp`, `--seed`.

In [ ]:
# IID
!python experiments/baseline/run_baseline.py --partition iid --rounds 20

In [ ]:
# Non-IID (Dirichlet). Smaller alpha = more skewed clients.
!python experiments/baseline/run_baseline.py --partition dirichlet --alpha 0.5 --rounds 20

## Inspect results

In [ ]:
import glob, json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

runs = sorted(glob.glob("results/*/summary.json"))
summaries = pd.DataFrame([json.load(open(p)) for p in runs])
display(summaries[["experiment_id", "partition", "alpha", "rounds", "final_accuracy", "best_accuracy", "total_wall_time_s", "device"]])

fig, ax = plt.subplots(figsize=(7, 4))
for p in runs:
    run_dir = os.path.dirname(p)
    df = pd.read_csv(f"{run_dir}/rounds.csv")
    ax.plot(df["round"], df["global_accuracy"], marker="o", label=os.path.basename(run_dir))
ax.set(xlabel="Round", ylabel="Test accuracy", title="FedAvg global accuracy")
ax.grid(alpha=0.3); ax.legend(fontsize=7)
plt.show()

latest = os.path.dirname(runs[-1])
display(Image(f"{latest}/class_distribution.png"))
pd.read_csv(f"{latest}/rounds.csv").head()

## Save results
Colab storage is wiped when the runtime ends. Download a zip, or mount Drive and copy `results/`.

In [ ]:
!zip -qr results.zip results
from google.colab import files
files.download("results.zip")

# Or keep them in Drive:
# from google.colab import drive
# drive.mount("/content/drive")
# !cp -r results /content/drive/MyDrive/carbon-aware-fl-results